<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/EventManagement/blob/main/MACCO_W1_CML_Noise_Adaptation_HF_COCO_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACCO W1 + CML → Progressive Gaussian-Noise Adaptation

**Colab notebook, no Google Drive.** Continue training from **our** Hugging Face checkpoint, `rifahnanjiba2/MACCO-Finetuned / MACCO-W1-CML-ViT-B-32.pt`.

**Data sources**
- **Checkpoint:** Hugging Face model repo (`hf_hub_download`).
- **Reviewed CML negatives:** [our GitHub file](https://github.com/rifahnanjiba02-arch/vision-language-composition/blob/main/results/cml_audit%20(1).csv), read directly at execution time (no manual upload).
- **Training annotations:** [MACCO authors' prepared MS COCO TSV](https://github.com/hiker-lw/MACCO), selected by the reviewed audit's `filepath` **and** `title`.
- **COCO 2014 training JPEGs:** [Hugging Face dataset mirror `ali-sh07/COCO-train2014`](https://huggingface.co/datasets/ali-sh07/COCO-train2014), fetching **only the needed JPEG files**, not the 13.5 GB full archive.
- **Outputs:** `/content/macco_w1_noise_outputs` — you can download files or optionally upload the final checkpoint to HF. The Colab runtime is temporary.

**Original W1+CML:** MCA=1, MIR=0.2, text reconstruction=25, image reconstruction=250, CML=1.0, margin=0.05. This experiment retains those weights and adds **Gaussian text-embedding noise**, while image masking stays as in MACCO. Noise is applied only to the *corrupted* text branch, at annotation-selected relation/attribute tokens.

**Checkpoint limitation verified from your original `MACCO_Modified_W1_CML` export cell:** `MACCO-W1-CML-ViT-B-32.pt` saves the **fine-tuned CLIP state dict only**. It does **not** save the MACCO prediction heads or learned mask tokens. Thus the encoder is fine-tuned, not random, but the training heads need a short masked warm-up. If you later upload a complete `model.state_dict()`, the loader can restore all heads and skip warm-up.

**Research note:** This is a pilot for *robustness adaptation*, not a guarantee of improved scores. The included CML sanity check uses already-seen audit images, so it is **not a held-out benchmark**. Use separate ARO/SugarCrepe evaluation later for reported results.

## 1 · Install imports and fetch MACCO source

Enable **Runtime → Change runtime type → GPU** first. We install the subset of dependencies needed by the authors' training pipeline.

In [1]:
%pip -q install "timm==1.0.15" ftfy regex "webdataset==0.2.111" braceexpand easydict PyYAML pytz wandb huggingface_hub pandas matplotlib tqdm requests

In [2]:
from pathlib import Path
import os, sys, subprocess
import torch

ROOT = Path('/content/macco_w1_noise')
AUTHOR_REPO = ROOT / 'MACCO'
ROOT.mkdir(parents=True, exist_ok=True)

if not (AUTHOR_REPO / '.git').exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--no-checkout',
                    'https://github.com/hiker-lw/MACCO.git', str(AUTHOR_REPO)],check=True)
if not (AUTHOR_REPO / 'src/training/data_macco.py').is_file():
    subprocess.run(['git','-C',str(AUTHOR_REPO),'sparse-checkout','set','--no-cone',
                    '/src/', '/mscoco_object_relation_attribute_with_phrase_bbox.tsv'], check=True)
    subprocess.run(['git','-C',str(AUTHOR_REPO),'checkout','HEAD'], check=True)

ANNOTATIONS = AUTHOR_REPO/'mscoco_object_relation_attribute_with_phrase_bbox.tsv'
assert ANNOTATIONS.is_file(), 'Author MACCO TSV was not downloaded.'
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before running.'
sys.path.insert(0, str(AUTHOR_REPO/'src'))
print('GPU:', torch.cuda.get_device_name(0))
print('MACCO author commit:',subprocess.check_output(
    ['git','-C',str(AUTHOR_REPO),'rev-parse','HEAD'],text=True).strip())

GPU: Tesla T4
MACCO author commit: 97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae


## 2 · Settings: Hugging Face model, GitHub CML review, COCO images



In [3]:
#@title W1+CML model, GitHub-reviewed negatives, HF COCO and training settings
HF_REPO_ID = 'rifahnanjiba2/MACCO-Finetuned'
HF_CHECKPOINT_FILE = 'MACCO-W1-CML-ViT-B-32.pt'
HF_REVISION = 'main'

GITHUB_REPO = 'rifahnanjiba02-arch/vision-language-composition'
GITHUB_REVISION = 'main'
GITHUB_AUDIT_PATH = 'results/cml_audit (1).csv'

HF_COCO_REPO = 'ali-sh07/COCO-train2014'
OUTPUT_DIR = Path('/content/macco_w1_noise_outputs')

SEED = 42
NUM_ROWS = 512             # All 512 audited W1+CML rows; set smaller only for smoke tests.
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_BATCHES_PER_EPOCH = 100  # None = all minibatches each epoch
WARMUP_MASK_BATCHES = 20   # Needed because this exported CLIP-only file lacks predictors
MASK_BATCH_PROB = 0.30     # Mix 30% original masked batches after warm-up
NOISE_START = 0.15
NOISE_END = 1.0
LR_CLIP = 5e-7
LR_HEADS = 1e-4
WEIGHT_DECAY = 0.01
MAX_COCO_WORKERS = 6

# Exactly your original W1 + CML candidate weights
W1_MCA = 1.0
W1_MIR = 0.2
W1_TEXT_RECON = 25.0
W1_IMAGE_RECON = 250.0
CML_WEIGHT = 1.0
CML_MARGIN = 0.05

# In-sample sanity diagnostic; NOT a published or held-out ARO result
RUN_AUDIT_DIAGNOSTIC = True
DIAGNOSTIC_ROWS = 32
DIAGNOSTIC_NOISE_SIGMA = 1.0

assert EPOCHS >= 1 and BATCH_SIZE >= 2 and GRAD_ACCUM >= 1
assert 0.0 <= MASK_BATCH_PROB <= 1.0 and 0 <= NOISE_START <= NOISE_END
assert NUM_ROWS >= BATCH_SIZE and WARMUP_MASK_BATCHES >= 0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT_DIR)

Outputs: /content/macco_w1_noise_outputs


## 3 · Download our fine-tuned W1+CML checkpoint from HF

The official MACCO **native CLIP** loader will consume this checkpoint in Section 8. We do **not** initialize from OpenAI CLIP.

In [4]:
from huggingface_hub import hf_hub_download

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

CHECKPOINT = Path(hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=HF_CHECKPOINT_FILE,
    revision=HF_REVISION,
    repo_type='model',
    token=HF_TOKEN,
))
print('Fine-tuned W1+CML checkpoint:', CHECKPOINT)
print('Size (MB):', round(CHECKPOINT.stat().st_size/1e6,1))

MACCO-W1-CML-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

MACCO-W1-CML-ViT-B-32.pt: downloading bytes:           |  0.00B            

Fine-tuned W1+CML checkpoint: /root/.cache/huggingface/hub/models--rifahnanjiba2--MACCO-Finetuned/snapshots/4d5fec101b210d5c9484fe9d040ed57e3bfa7d19/MACCO-W1-CML-ViT-B-32.pt
Size (MB): 605.2


## 4 · Download `results/cml_audit (1).csv` directly from GitHub

The **existing human review** is reused as-is. We do **not** regenerate/reapprove negatives; only rows explicitly marked `approved=True` receive a CML penalty.

In [5]:
from urllib.parse import quote
import pandas as pd
import requests

GH_URL = (f'https://raw.githubusercontent.com/{GITHUB_REPO}/'
          f'{quote(GITHUB_REVISION, safe="")}/'
          f'{quote(GITHUB_AUDIT_PATH, safe="/")}')
AUDIT_CSV = ROOT / 'cml_audit_reviewed.csv'
response = requests.get(GH_URL, timeout=90)
response.raise_for_status()
assert response.content.startswith(b'row_id,'), (
    'GitHub download did not return the expected audit CSV; check the URL.')
AUDIT_CSV.write_bytes(response.content)
audit = pd.read_csv(AUDIT_CSV, keep_default_na=False)
required = {'row_id','filepath','title','candidate_negative','approved'}
assert required.issubset(audit.columns), f'Missing audit columns: {required - set(audit.columns)}'
audit['approved'] = audit['approved'].astype(str).str.strip().str.lower().isin(
    ['true','1','yes','y'])
assert not audit.row_id.duplicated().any(), 'Duplicate audit row_id entries.'
assert not audit[['filepath','title']].duplicated().any(), 'Repeated file+caption in audit; inspect first.'
assert not (audit.approved & audit.candidate_negative.astype(str).str.strip().eq('')).any(), (
    'Approved row has no candidate negative.')
assert not (audit.approved & (audit.candidate_negative.str.strip() == audit.title.str.strip())).any(), (
    'An approved negative is identical to its positive caption.')
print('GitHub audit:', GH_URL)
print('Reviewed rows:', len(audit))
print('Approved negative captions:', int(audit.approved.sum()))
display(audit[['row_id','title','candidate_negative','approved']].head(5))

GitHub audit: https://raw.githubusercontent.com/rifahnanjiba02-arch/vision-language-composition/main/results/cml_audit%20%281%29.csv
Reviewed rows: 512
Approved negative captions: 107


,row_id,title,candidate_negative,approved
0,0,some street signs sitting by a wall,,False
1,1,Two surfers at the beach walking into the waves.,,False
2,2,Antique cars on the grass in front of an antiq...,Antique cars on the grass behind an antique tr...,True
3,3,A man in a blue shirt holding a white plate wi...,A man in a red shirt holding a white plate wit...,True
4,4,A group of men standing around a batting cage.,,False


## 5 · Rebuild the original training TSV from the approved audit

The audit CSV intentionally has only review fields. MACCO's own dataset loader also requires prepared **object, relation, attribute and bounding-box annotations**. We join the exact audited `(filepath, title)` pairs back to the authors' 10-column TSV rather than inventing annotations.

In [6]:
import numpy as np

original = pd.read_csv(ANNOTATIONS, sep='\t')
original_cols = list(original.columns)
need_original = {'filepath','title','neg_caption','neg_image','object_list',
                 'relation_list','attribute_list','relation_phrase_list',
                 'attribute_phrase_list','relation_attribute_phrase_bbox_list'}
assert need_original.issubset(original_cols), 'Author annotation schema changed.'
selected_audit = audit.sort_values('row_id').head(NUM_ROWS).copy()
if original.duplicated(['filepath','title']).any():
    print('Note: removing repeated identical filepath/title rows from author annotations.')
    original = original.drop_duplicates(['filepath','title'], keep='first')
train_rows = selected_audit.merge(original, on=['filepath','title'],
                                  how='left', validate='one_to_one', sort=False)
assert train_rows['relation_attribute_phrase_bbox_list'].notna().all(), (
    'An audited filepath/title is absent from the authors TSV. Check both sources.')
train_rows = train_rows.sort_values('row_id').reset_index(drop=True)
# Non-approved rows receive the original title as their dummy negative, exactly as in W1+CML.
train_rows['neg_caption'] = [repr([neg if bool(ok) else pos])
    for ok,neg,pos in zip(train_rows.approved, train_rows.candidate_negative, train_rows.title)]
TRAIN_TSV = AUTHOR_REPO / 'cml_noise_train.tsv'
train_rows[original_cols].to_csv(TRAIN_TSV, sep='\t', index=False)
assert int(train_rows.approved.sum()) >= 10, 'Too few approved rows for a meaningful CML pilot.'
assert all(str(x).startswith('coco/train2014/') for x in train_rows.filepath), (
    'Expected authors COCO 2014 train2014 paths.')
print('Original author TSV rows:',len(original))
print('Matched original reviewed W1+CML rows:',len(train_rows))
print('Approved CML pairs in this training set:',int(train_rows.approved.sum()))
print('Output TSV:',TRAIN_TSV)
display(train_rows[['row_id','filepath','title','candidate_negative','approved']].head(4))

Original author TSV rows: 109468
Matched original reviewed W1+CML rows: 512
Approved CML pairs in this training set: 107
Output TSV: /content/macco_w1_noise/MACCO/cml_noise_train.tsv


,row_id,filepath,title,candidate_negative,approved
0,0,coco/train2014/COCO_train2014_000000052826.jpg,some street signs sitting by a wall,,False
1,1,coco/train2014/COCO_train2014_000000401846.jpg,Two surfers at the beach walking into the waves.,,False
2,2,coco/train2014/COCO_train2014_000000073215.jpg,Antique cars on the grass in front of an antiq...,Antique cars on the grass behind an antique tr...,True
3,3,coco/train2014/COCO_train2014_000000332204.jpg,A man in a blue shirt holding a white plate wi...,A man in a red shirt holding a white plate wit...,True


## 6 · Download **only selected COCO 2014 JPEGs** from Hugging Face

This uses the same HF mirror (`ali-sh07/COCO-train2014`) as your earlier W1+CML workflow. Files in that mirror are at its root; the author's loader expects `datasets/coco/train2014/COCO_....jpg`. `hf_hub_download()` caches them, then we copy them into the correct author dataset layout. Missing files cause a clear failure instead of silently changing the training rows.

In [9]:

#@title Download required COCO images from Hugging Face (fixed)

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
import pandas as pd
import io

data_dir = AUTHOR_REPO / "datasets"
data_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

unique_paths = sorted(set(train_rows.filepath.astype(str)))
HF_FULL_COCO = "AbdoTW/COCO_2014"

def valid_image(path):
    if not path.is_file():
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        path.unlink(missing_ok=True)
        return False

# Reuse images downloaded in previous attempts
manifest = []
missing = {}

for rel in unique_paths:
    target = data_dir / rel
    if valid_image(target):
        manifest.append({
            "filepath": rel,
            "bytes": target.stat().st_size,
            "source": "local-cache"
        })
    else:
        image_id = int(Path(rel).stem.split("_")[-1])
        missing[image_id] = rel

print("Already available:", len(manifest))
print("Still needed:", len(missing))

if missing:
    print("Loading missing COCO images from:", HF_FULL_COCO)

    # Stream only matching IDs from the full HF dataset.
    # Parquet filtering avoids loading the entire dataset into RAM.
    ds = load_dataset(
        HF_FULL_COCO,
        split="train",
        streaming=True,
        filters=[("id", "in", list(missing.keys()))],
        token=HF_TOKEN
    )

    for sample in tqdm(ds, desc="Retrieving missing COCO images"):
        image_id = int(sample["id"])

        if image_id not in missing:
            continue

        rel = missing[image_id]
        target = data_dir / rel
        target.parent.mkdir(parents=True, exist_ok=True)

        image = sample["image"]

        if isinstance(image, dict):
            if image.get("bytes") is not None:
                image = Image.open(io.BytesIO(image["bytes"]))
            else:
                image = Image.open(image["path"])

        # Save as a valid JPEG
        with image:
            image.convert("RGB").save(
                target, format="JPEG", quality=95
            )

        if not valid_image(target):
            raise RuntimeError(f"Invalid image: {rel}")

        manifest.append({
            "filepath": rel,
            "bytes": target.stat().st_size,
            "source": "hf-full-coco"
        })

        del missing[image_id]

        if not missing:
            break

if missing:
    raise RuntimeError(
        f"{len(missing)} images still missing. "
        f"Examples: {list(missing.values())[:5]}"
    )

manifest_csv = OUTPUT_DIR / "coco_hf_manifest.csv"

pd.DataFrame(manifest).sort_values(
    "filepath"
).to_csv(manifest_csv, index=False)

print("\nSuccessfully retrieved:", len(manifest), "images")
print(pd.DataFrame(manifest)["source"].value_counts())
print("Training JPEG folder:", data_dir / "coco/train2014")
print("Manifest:", manifest_csv)


Already available: 60
Still needed: 452
Loading missing COCO images from: AbdoTW/COCO_2014


README.md:   0%|          | 0.00/808 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Retrieving missing COCO images: 0it [00:00, ?it/s]

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/AbdoTW/COCO_2014/resolve/3c2e0bdcb4c385ba2e7d010ea9067f7a97300385/data/train-00024-of-00028.parquet
Retrying in 1s [Retry 1/5].



Successfully retrieved: 512 images
source
hf-full-coco    452
local-cache      60
Name: count, dtype: int64
Training JPEG folder: /content/macco_w1_noise/MACCO/datasets/coco/train2014
Manifest: /content/macco_w1_noise_outputs/coco_hf_manifest.csv


## 7 · Add selectable text-noise corruption to the authors' MACCO source

We patch only the **training repository copy in this Colab runtime**. Original masking remains available. As in our earlier modified notebook, we fix the authors' method binding for modern Python. The corruption indices still come from the original scene-graph annotations. Image masking is unchanged.

In [10]:
# Idempotent source patch; fail if the author's upstream implementation changed.
model_path = AUTHOR_REPO/'src/open_clip_code/MACCO_variant/MACCO_text_image.py'
src = model_path.read_text()
for method, target in [('clip_encode_text','self.CLIP'),
                       ('clip_encode_image','self.CLIP'),
                       ('clip_vit_forward','self.CLIP.visual'),
                       ('clip_forward','self.CLIP')]:
    original_bind = f'self.{method}.__get__({target})'
    fixed_bind = f'type(self).{method}.__get__({target}, type({target}))'
    if original_bind in src:
        src = src.replace(original_bind,fixed_bind,1)
    else:
        assert fixed_bind in src, f'Cannot locate method binding for {method}'

# Normalize the two physical lines exactly to avoid whitespace-only match problems.
needle = "        x_full_mask = x_full_mask * (~key_mask[:, :, None]) + " + chr(92) + "\n" + \
         "                text_mask_token[None, None, :] * key_mask[:, :, None]"
replacement = """        # W1_CML_NOISE_PATCH: keep the clean branch and selectively corrupt
        # selected compositional tokens only in the second (corrupted) branch.
        mode = getattr(self, 'text_corruption_mode', 'mask')
        if mode == 'noise':
            scale = x_full_mask[:text.shape[0]].detach().float().std().clamp_min(1e-6)
            scale = scale.to(dtype=x_full_mask.dtype)
            sigma = float(getattr(self, 'text_noise_strength', 0.0))
            x_full_mask = x_full_mask + (sigma * scale * torch.randn_like(x_full_mask)) * key_mask.unsqueeze(-1)
        elif mode == 'mask':
            x_full_mask = x_full_mask * (~key_mask[:, :, None]) + \\
                    text_mask_token[None, None, :] * key_mask[:, :, None]
        else:
            raise ValueError(f'Unknown corruption mode: {mode}')"""
if 'W1_CML_NOISE_PATCH' not in src:
    assert src.count(needle)==1, 'MACCO mask operation changed; inspect source before patching.'
    src = src.replace(needle, replacement, 1)
model_path.write_text(src)
compile(src,str(model_path),'exec')
print('Patched MACCO text masking with mask/noise toggle (no image change).')

Patched MACCO text masking with mask/noise toggle (no image change).


## 8 · Load fine-tuned encoder and reconstruct MACCO training model

We use **`create_model_and_transforms('ViT-B-32', pretrained=CHECKPOINT)`**, as requested. The author's exported W1+CML `.pt` contains only CLIP weights; the full MACCO wrapper is then attached for training. A future full checkpoint can restore the heads automatically.

In [11]:
import math, random, json, gc
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from open_clip_code import create_model_and_transforms, tokenize
from open_clip_code.MACCO_variant.MACCO_text_image import MACCO_CLIP_text_image
from open_clip_code.loss_fn.MACCO_CLIP_loss_text_image_raw_space import (
    macco_clip_loss_text_image, compute_contrast_loss)
from training.data_macco import get_data
from training.utils import create_transform_with_tracking

DEVICE = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
from training.params import parse_args
# MACCO's YAML does not include all data-loader defaults. Start from its parser.
_saved_argv = sys.argv[:]
try:
    sys.argv = [sys.argv[0], '--model', 'ViT-B-32']
    args = parse_args()
finally:
    sys.argv = _saved_argv
for _key, _value in yaml.safe_load((AUTHOR_REPO/'src/config/MACCO_CLIP.yaml').read_text()).items():
    setattr(args, _key, _value)
# Override author defaults to reproduce original W1+CML objective.
args.model = 'ViT-B-32'
args.train_data = TRAIN_TSV.name
args.dataset_type = 'csv'
args.batch_size = BATCH_SIZE
args.workers = 2
args.mask_text_image = True
args.mask_image = False
args.mask_text = False
args.add_neg_sample = False
args.random_mask_text = False
args.random_mask_image = False
args.distributed = False
args.world_size = 1
args.imagenet_val = None
args.imagenet_v2 = None
args.val_data = None
args.contrast_loss_w = W1_MCA
args.intra_modal_contrast_loss_w = W1_MIR
args.mask_text_loss_w = W1_TEXT_RECON
args.mask_image_loss_w = W1_IMAGE_RECON
args.cml_weight = CML_WEIGHT
args.cml_margin = CML_MARGIN

# Auto-detect CLIP-only versus full MACCO checkpoint.
raw = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
state = raw.get('state_dict',raw) if isinstance(raw,dict) else raw
assert isinstance(state,dict), 'Checkpoint should be a PyTorch model state dict.'
state = {k.removeprefix('module.'):v for k,v in state.items()}
full_model_state = any(k.startswith('CLIP.') for k in state)
if full_model_state:
    # Unpack into a temporary file to use the same native CLIP loader.
    tmp = ROOT/'clip_extracted_for_native_loader.pt'
    torch.save({k.removeprefix('CLIP.'):v for k,v in state.items()
                if k.startswith('CLIP.')}, tmp)
    native_checkpoint = tmp
else:
    native_checkpoint = CHECKPOINT

clip_model, _, preprocess_val = create_model_and_transforms(
    'ViT-B-32', pretrained=str(native_checkpoint), device='cpu', precision='fp32')
model = MACCO_CLIP_text_image(clip_model,args)
if full_model_state:
    model.load_state_dict(state, strict=True)
    print('Restored full MACCO + CLIP checkpoint, including reconstruction heads.')
    actual_warmup = 0
else:
    print('Restored original fine-tuned W1+CML CLIP encoder.')
    print('Your W1+CML export DOES NOT include trained MACCO predictor heads.')
    print('New reconstruction heads and learned mask tokens will be trained.')
    actual_warmup = WARMUP_MASK_BATCHES
model = model.to(DEVICE)
model.CLIP.text_corruption_mode = 'mask'
model.CLIP.text_noise_strength = 0.0
print('Checkpoint type:', 'full MACCO' if full_model_state else 'CLIP-only W1+CML')
print('W1 loss weights / CML margin:', [W1_MCA,W1_MIR,W1_TEXT_RECON,W1_IMAGE_RECON,CML_WEIGHT,CML_MARGIN])
del raw, state
if full_model_state and native_checkpoint.exists() and native_checkpoint!=CHECKPOINT:
    native_checkpoint.unlink(missing_ok=True)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Restored original fine-tuned W1+CML CLIP encoder.
Your W1+CML export DOES NOT include trained MACCO predictor heads.
New reconstruction heads and learned mask tokens will be trained.
Checkpoint type: CLIP-only W1+CML
W1 loss weights / CML margin: [1.0, 0.2, 25.0, 250.0, 1.0, 0.05]


## 9 · Use the authors' original scene-graph masking flags and reconstruction loss

The positions flagged for masking become **noise positions** in noise-mode batches. The original `neg_caption` field supplies the exact reviewed CML negatives.

In [12]:
train_preprocess, get_transform_params = create_transform_with_tracking(image_size=224)
data = get_data(args,(train_preprocess,preprocess_val),epoch=0,
                get_transform_params=get_transform_params)
train_loader = data['train'].dataloader
train_dataset = train_loader.dataset
criterion = macco_clip_loss_text_image(args)
assert len(train_loader)>0, 'No training batches: check BATCH_SIZE or data paths.'
preview = next(iter(train_loader))
images, texts, neg_tokens, comp_flag, obj_flag, image_flag = preview
assert texts.shape == comp_flag.shape
assert image_flag.shape[1] == (224 // model.patch_size)**2 + 1
print('Training rows:', len(train_dataset), '| batches/epoch:',len(train_loader))
print('Input:',tuple(images.shape), '| annotated concept tokens:',int(comp_flag.sum()))
print('Approved CML rows (expected nonzero):',int(train_rows.approved.sum()))

Training rows: 512 | batches/epoch: 128
Input: (4, 3, 224, 224) | annotated concept tokens: 9
Approved CML rows (expected nonzero): 107


## 10 · Preserve W1+CML during noise adaptation

CML is the same **cosine-margin hinge** you added earlier, evaluated only for approved negatives. For noisy batches the contrastive component uses clean image–clean text and clean image–noisy text as **positive** pairs, avoiding clean/noisy captions competing as negatives. Masked batches use the original MACCO contrastive term. All W1 reconstruction and intra-modal loss weights remain unchanged.

In [13]:
def objective_w1_cml(output, texts, negative_texts, mode):
    losses = criterion(output)  # Original four MACCO objectives
    if mode == 'noise':
        img = output['img_feat_cls_token']
        clean_text = output['text_feat_cls_token']
        noisy_text = output['masked_text_feat_cls_token']
        scale = output['logit_scale']
        losses['contrast_loss'] = 0.5 * (
            compute_contrast_loss(img,clean_text,scale,img.device) +
            compute_contrast_loss(img,noisy_text,scale,img.device))
    # Exactly the original W1+CML contribution: approved negative captions only.
    empty_mask = torch.zeros_like(negative_texts,dtype=torch.bool)
    neg_raw, _, _, _ = model.CLIP.encode_text(
        negative_texts,empty_mask,model.text_mask_token)
    neg = F.normalize(neg_raw,dim=-1)
    image_embed = output['img_feat_cls_token']
    positive_text = output['text_feat_cls_token']
    positive_cos = (image_embed * positive_text).sum(dim=-1)
    negative_cos = (image_embed * neg).sum(dim=-1)
    has_approved_negative = (negative_texts!=texts).any(dim=1)
    hinge = F.relu(CML_MARGIN + negative_cos - positive_cos)
    cml = hinge[has_approved_negative].mean() if has_approved_negative.any() else hinge.sum()*0.0
    losses['cml_loss'] = cml
    losses['approved_in_batch'] = has_approved_negative.sum().detach()
    losses['total_loss'] = (
        W1_MCA*losses['contrast_loss'] +
        W1_MIR*losses['intra_modal_contrast_loss'] +
        W1_TEXT_RECON*losses['mask_loss_text'] +
        W1_IMAGE_RECON*losses['mask_loss_image'] +
        CML_WEIGHT*cml)
    return losses

@torch.no_grad()
def check_single_batch(mode, sigma=0.0):
    model.eval()
    model.CLIP.text_corruption_mode = mode
    model.CLIP.text_noise_strength = sigma
    x,t,n,tf,of,im = preview
    count = min(2,len(t))
    x,t,n,tf,im = (a[:count].to(DEVICE) for a in (x,t,n,tf,im))
    with torch.autocast('cuda',dtype=torch.float16):
        out = model(x,t,im,tf)
        loss = objective_w1_cml(out,t,n,mode)
    report = {k:round(float(v.detach().cpu()),4) for k,v in loss.items()}
    assert all(np.isfinite(v) for v in report.values()),report
    return report

print('Mask smoke test:',check_single_batch('mask'))
print('Noise smoke test:',check_single_batch('noise',NOISE_START))

Mask smoke test: {'contrast_loss': 0.2978, 'intra_modal_contrast_loss': 0.1589, 'mask_loss_text': 10.8533, 'mask_loss_image': 1.0002, 'total_loss': 521.7383, 'cml_loss': 0.0205, 'approved_in_batch': 1.0}
Noise smoke test: {'contrast_loss': 0.0, 'intra_modal_contrast_loss': 0.1588, 'mask_loss_text': 10.8656, 'mask_loss_image': 1.0002, 'total_loss': 521.748, 'cml_loss': 0.0205, 'approved_in_batch': 1.0}


## 11 · Optional diagnostic: CML true-vs-approved-negative ranking

Measured on **a fixed subset of the already-reviewed W1 training images**. This is an internal before/after sanity check, **not a generalization metric**. Image transforms and random noise are seeded identically before and after.

In [14]:
approved_positions = np.flatnonzero(train_rows.approved.to_numpy())[:DIAGNOSTIC_ROWS].tolist()

@torch.no_grad()
def audit_ranking_diagnostic(sigma):
    assert approved_positions, 'No approved rows in training set.'
    random.seed(SEED+202); np.random.seed(SEED+202)
    torch.manual_seed(SEED+202); torch.cuda.manual_seed_all(SEED+202)
    diagnostic_data = DataLoader(Subset(train_dataset,approved_positions),
                                 batch_size=min(8,BATCH_SIZE),shuffle=False,num_workers=0)
    model.eval()
    model.CLIP.text_corruption_mode = 'noise'
    model.CLIP.text_noise_strength = sigma
    correct,total = 0,0
    for imgs,ids,neg_ids,text_flags,_,im_flags in diagnostic_data:
        imgs,ids,neg_ids,text_flags = [v.to(DEVICE) for v in
                                        (imgs,ids,neg_ids,text_flags)]
        zero_im = torch.zeros((len(imgs),(224//model.patch_size)**2+1),
                               device=DEVICE,dtype=torch.bool)
        img_cls,*_ = model.CLIP.encode_image(imgs,zero_im,model.img_mask_token)
        _,_,corrupted_cls,_ = model.CLIP.encode_text(
            ids,text_flags,model.text_mask_token)
        zeros_text = torch.zeros_like(neg_ids,dtype=torch.bool)
        neg_cls,*_ = model.CLIP.encode_text(neg_ids,zeros_text,model.text_mask_token)
        img_cls = F.normalize(img_cls.float(),dim=-1)
        corrupted_cls = F.normalize(corrupted_cls.float(),dim=-1)
        neg_cls = F.normalize(neg_cls.float(),dim=-1)
        positive_score = (img_cls*corrupted_cls).sum(-1)
        negative_score = (img_cls*neg_cls).sum(-1)
        correct += int((positive_score>negative_score).sum())
        total += len(imgs)
    return {'sigma':sigma,'ranking_accuracy':correct/max(1,total),'n':total}

before_diag=[]
if RUN_AUDIT_DIAGNOSTIC:
    before_diag=[audit_ranking_diagnostic(0.0),
                 audit_ranking_diagnostic(DIAGNOSTIC_NOISE_SIGMA)]
    print('Before adaptation (already-seen training images):')
    display(pd.DataFrame(before_diag))
model.CLIP.text_corruption_mode='mask'

Before adaptation (already-seen training images):


,sigma,ranking_accuracy,n
0,0.0,0.8125,32
1,1.0,0.5000,32


## 12 · Train: masked warm-up → progressively stronger Gaussian noise

Each optimizer update adjusts the fine-tuned encoder and the MACCO predictors. After warm-up, we keep some mask batches to reduce catastrophic forgetting. We retain the **W1+CML** margin loss and save a **full** MACCO checkpoint after every epoch. Settings above are an initial smoke/pilot run.

In [15]:
from tqdm.auto import tqdm

clip_ids={id(p) for p in model.CLIP.parameters()}
clip_params=[p for p in model.CLIP.parameters() if p.requires_grad]
head_params=[p for p in model.parameters() if id(p) not in clip_ids and p.requires_grad]
optimizer = torch.optim.AdamW([
    {'params':clip_params,'lr':LR_CLIP},
    {'params':head_params,'lr':LR_HEADS},
],weight_decay=WEIGHT_DECAY)
scaler=torch.amp.GradScaler('cuda', enabled=True)
steps_per_epoch=min(len(train_loader),MAX_BATCHES_PER_EPOCH or len(train_loader))
total_steps=steps_per_epoch*EPOCHS
assert actual_warmup < total_steps, 'Warmup would consume all training steps.'

def choose_mode(step):
    if step < actual_warmup:
        return 'mask', 0.0
    fraction = (step-actual_warmup)/max(1,total_steps-actual_warmup-1)
    sigma = NOISE_START+(NOISE_END-NOISE_START)*min(1.0,fraction)
    mode = 'mask' if random.random()<MASK_BATCH_PROB else 'noise'
    return mode, (0.0 if mode=='mask' else sigma)

history=[]
global_step=0
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    metrics = {k:0.0 for k in ['total_loss','contrast_loss','intra_modal_contrast_loss',
                               'mask_loss_text','mask_loss_image','cml_loss']}
    approved_seen=0
    for batch_i,batch in enumerate(tqdm(train_loader,total=steps_per_epoch,
                                        desc=f'W1+CML noise epoch {epoch+1}/{EPOCHS}')):
        if batch_i>=steps_per_epoch:
            break
        imgs,toks,negative,flags,_,patch_flags=[v.to(DEVICE,non_blocking=True) for v in batch]
        mode,sigma=choose_mode(global_step)
        model.CLIP.text_corruption_mode=mode
        model.CLIP.text_noise_strength=sigma
        with torch.autocast('cuda',dtype=torch.float16):
            out=model(imgs,toks,patch_flags,flags)
            losses=objective_w1_cml(out,toks,negative,mode)
            # Correct scaling on the last short accumulation window.
            win_start=(batch_i//GRAD_ACCUM)*GRAD_ACCUM
            win_size=min(GRAD_ACCUM,steps_per_epoch-win_start)
            scaled_loss=losses['total_loss']/win_size
        if not torch.isfinite(scaled_loss):
            raise RuntimeError(f'Non-finite loss at epoch {epoch+1}, batch {batch_i}.')
        scaler.scale(scaled_loss).backward()
        if (batch_i+1)%GRAD_ACCUM==0 or (batch_i+1)==steps_per_epoch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                model.CLIP.logit_scale.clamp_(0,math.log(100))
        for name in metrics:
            metrics[name]+=float(losses[name].detach().cpu())
        approved_seen+=int(losses['approved_in_batch'])
        global_step+=1
    epoch_record={'epoch':epoch+1,'batches':steps_per_epoch,'approved_pairs_seen':approved_seen,
                  'noise_sigma_last_batch':sigma,
                  **{k:v/steps_per_epoch for k,v in metrics.items()}}
    history.append(epoch_record)
    out_file=OUTPUT_DIR/f'W1_CML_noise_full_epoch_{epoch+1}.pt'
    torch.save({'epoch':epoch+1,'state_dict':model.state_dict(),
                'history':history,
                'settings':{'w1_weights':[W1_MCA,W1_MIR,W1_TEXT_RECON,W1_IMAGE_RECON],
                            'cml_weight':CML_WEIGHT,'cml_margin':CML_MARGIN,
                            'hf_checkpoint':f'{HF_REPO_ID}/{HF_CHECKPOINT_FILE}',
                            'hf_coco_repo':HF_COCO_REPO,
                            'github_audit':GH_URL}},out_file)
    print('Saved FULL MACCO weights:',out_file.name)
    display(pd.DataFrame([epoch_record]))

# CLIP-only export for your existing MACCO Table 1 evaluation notebook.
clip_export=OUTPUT_DIR/'MACCO-W1-CML-Noise-ViT-B-32.pt'
torch.save(model.CLIP.state_dict(),clip_export)
print('Saved CLIP-only weights:',clip_export)

pd.DataFrame(history).to_csv(OUTPUT_DIR/'noise_training_history.csv',index=False)
model.CLIP.text_corruption_mode='mask'
model.CLIP.text_noise_strength=0.0

W1+CML noise epoch 1/3:   0%|          | 0/100 [00:00<?, ?it/s]

Saved FULL MACCO weights: W1_CML_noise_full_epoch_1.pt


,epoch,batches,approved_pairs_seen,noise_sigma_last_batch,total_loss,contrast_loss,intra_modal_contrast_loss,mask_loss_text,mask_loss_image,cml_loss
0,1,100,80,0.390681,504.797014,0.168302,0.175056,10.230538,0.995228,0.023266


W1+CML noise epoch 2/3:   0%|          | 0/100 [00:00<?, ?it/s]

Saved FULL MACCO weights: W1_CML_noise_full_epoch_2.pt


,epoch,batches,approved_pairs_seen,noise_sigma_last_batch,total_loss,contrast_loss,intra_modal_contrast_loss,mask_loss_text,mask_loss_image,cml_loss
0,2,100,77,0.695341,460.200689,0.067888,0.146895,8.587676,0.981553,0.023177


W1+CML noise epoch 3/3:   0%|          | 0/100 [00:00<?, ?it/s]

Saved FULL MACCO weights: W1_CML_noise_full_epoch_3.pt


,epoch,batches,approved_pairs_seen,noise_sigma_last_batch,total_loss,contrast_loss,intra_modal_contrast_loss,mask_loss_text,mask_loss_image,cml_loss
0,3,100,81,0.0,420.54958,0.077884,0.129979,7.176477,0.964041,0.023487


Saved CLIP-only weights: /content/macco_w1_noise_outputs/MACCO-W1-CML-Noise-ViT-B-32.pt


## 13 · Compare fixed CML audit diagnostic before/after

Because the original checkpoint was already trained using these audit examples, this diagnostic is **only a training-set consistency check**, not a defensible ARO or clean/noisy generalization claim. Test on held-out ARO/SugarCrepe with identical noise settings for research conclusions.

In [16]:
if RUN_AUDIT_DIAGNOSTIC:
    after_diag=[audit_ranking_diagnostic(0.0),
                audit_ranking_diagnostic(DIAGNOSTIC_NOISE_SIGMA)]
    before=pd.DataFrame(before_diag).rename(columns={'ranking_accuracy':'before'})
    after=pd.DataFrame(after_diag).rename(columns={'ranking_accuracy':'after'})
    comparison=before.merge(after,on=['sigma','n'])
    comparison['change_pp']=(comparison['after']-comparison['before'])*100.0
    display(comparison)
    comparison.to_csv(OUTPUT_DIR/'reviewed_cml_audit_diagnostic.csv',index=False)
else:
    print('Diagnostic disabled.')
print('Output files:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' ',p.name,round(p.stat().st_size/1e6,1),'MB')

,sigma,before,n,after,change_pp
0,0.0,0.8125,32,0.90625,9.375
1,1.0,0.5000,32,0.59375,9.375


Output files:
  MACCO-W1-CML-Noise-ViT-B-32.pt 605.2 MB
  W1_CML_noise_full_epoch_1.pt 777.2 MB
  W1_CML_noise_full_epoch_2.pt 777.2 MB
  W1_CML_noise_full_epoch_3.pt 777.2 MB
  coco_hf_manifest.csv 0.0 MB
  noise_training_history.csv 0.0 MB
  reviewed_cml_audit_diagnostic.csv 0.0 MB


## 14 · Save your results (no Drive)

**Important:** `/content` disappears when the Colab runtime resets. Download the `.pt` files you need, or optionally push **your new** checkpoint to Hugging Face. The upload cell is **off by default** and will never overwrite your original W1+CML file.

In [17]:
#@title Optional: upload the NEW CLIP-only noise-adapted model to HF (disabled by default)
UPLOAD_NOISE_MODEL_TO_HF = False
HF_NEW_FILENAME = 'MACCO-W1-CML-Noise-ViT-B-32.pt'   # New filename; original stays untouched.

if UPLOAD_NOISE_MODEL_TO_HF:
    from huggingface_hub import HfApi
    if HF_NEW_FILENAME == HF_CHECKPOINT_FILE:
        raise ValueError('Refusing to overwrite the original fine-tuned checkpoint.')
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=str(clip_export),
                                    path_in_repo=HF_NEW_FILENAME,
                                    repo_id=HF_REPO_ID, repo_type='model')
    print('Uploaded NEW CLIP-only checkpoint:', HF_REPO_ID, '/', HF_NEW_FILENAME)
else:
    print('HF upload disabled. Results remain in',OUTPUT_DIR)

# Optional browser downloads from the Colab Files panel:
#   files/download from OUTPUT_DIR, e.g. MACCO-W1-CML-Noise-ViT-B-32.pt
#   W1_CML_noise_full_epoch_3.pt preserves the COMPLETE adapted training model.

HF upload disabled. Results remain in /content/macco_w1_noise_outputs


In [18]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/MACCO_W1_CML_Noise_Results",
    "zip",
    "/content/macco_w1_noise_outputs"
)

files.download("/content/MACCO_W1_CML_Noise_Results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### What this experiment does and does not prove

- **Does train:** gradients update the fine-tuned W1+CML CLIP encoder and the reinitialized MACCO reconstruction predictors, with progressively stronger **text-embedding** noise and the reviewed CML margin objective.
- **Does not use Drive:** HF model + HF JPEGs + your GitHub audit + author TSV; only `/content` storage.
- **Not the original full W1 training state:** your exported file contains the CLIP encoder state dict. The new **full** checkpoints created here do include MACCO heads for subsequent adaptation.
- **Not an ARO measurement:** the audit ranking diagnostic is in-sample. Run your original Table 1 evaluation notebook for clean benchmarks; later add a matched noisy benchmark evaluation, preserving the exact noise definition/seed.
- **For longer runs:** increase `MAX_BATCHES_PER_EPOCH`, `EPOCHS`, and the number of training rows only if suitable **human-reviewed** negatives remain available. Keep the original checkpoint separate from adapted outputs.

Inputs: [your reviewed audit](https://github.com/rifahnanjiba02-arch/vision-language-composition/blob/main/results/cml_audit%20(1).csv) · [HF COCO JPEG mirror](https://huggingface.co/datasets/ali-sh07/COCO-train2014) · [original MACCO code](https://github.com/hiker-lw/MACCO).